# Web scraping Les Echos avec Selenium

## Objectif

Rechercher les articles correspondant au terme **ia**, ouvrir chaque article et récupérer :

- son titre ;
- son petit résumé lorsqu'il existe ;
- son URL.


# Étape 1 - Installer les bibliothèques


In [1]:
# %pip install selenium pandas

# Étape 2 - Importer les bibliothèques


In [ ]:
from pathlib import Path

import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

TIMEOUT = 15
MAX_ARTICLES = 20  
NOMBRE_PAGES = 1   

# Étape 3 - Préparer les fonctions utiles

## Fonctions de recherche d'éléments


In [3]:
def trouver_premier(driver, selecteurs):
    """Renvoie le premier élément trouvé parmi plusieurs sélecteurs."""
    for by, valeur in selecteurs:
        elements = driver.find_elements(by, valeur)
        if elements:
            return elements[0]
    return None


def texte_premier(driver, selecteurs, valeur_par_defaut=""):
    """Renvoie le texte du premier élément trouvé."""
    element = trouver_premier(driver, selecteurs)
    return element.text.strip() if element else valeur_par_defaut


def attribut_premier(driver, selecteurs, attribut, valeur_par_defaut=""):
    """Renvoie un attribut du premier élément trouvé."""
    element = trouver_premier(driver, selecteurs)
    if not element:
        return valeur_par_defaut
    return (element.get_attribute(attribut) or valeur_par_defaut).strip()

# Étape 4 - Ouvrir Chrome et accéder au site

## Configuration du navigateur



In [4]:
options = webdriver.ChromeOptions()
options.binary_location = r"C:\Program Files\Google\Chrome\Application\chrome.exe"
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, TIMEOUT)

driver.get("https://www.lesechos.fr/")

if driver.title == "Access Denied":
    driver.quit()
    raise RuntimeError("Les Echos a refusé cette session Chrome. Relancer le navigateur sans mode headless.")

print("Page ouverte :", driver.title)

Page ouverte : Les Echos : actualités en direct, Économie, Finance, Marchés, Politique, Entreprises, Start-up - Les Echos


# Étape 5 - Accepter les cookies




In [5]:
try:
    bouton_accepter = WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.XPATH, "//button[normalize-space()='Accepter']"))
    )
    bouton_accepter.click()
    print("Cookies acceptés.")
except TimeoutException:
    print("Le bandeau de cookies n'est pas affiché ou le choix a déjà été enregistré.")

Cookies acceptés.


# Étape 6 - Rechercher les articles sur l'IA

## Ouvrir la recherche


In [6]:
bouton_recherche = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "button[aria-label='Recherche']"))
)
bouton_recherche.click()

champ_recherche = wait.until(
    EC.visibility_of_element_located((By.CSS_SELECTOR, "input[name='query']"))
)
champ_recherche.clear()
champ_recherche.send_keys("ia")
champ_recherche.send_keys(Keys.ENTER)

wait.until(EC.url_contains("/recherche?search=ia"))
print("Page de résultats :", driver.current_url)

Page de résultats : https://www.lesechos.fr/recherche?search=ia&searchType=posts


# Étape 7 - Récupérer les liens des articles

## Parcourir les pages de résultats


In [7]:
def recuperer_liens_articles(driver, nombre_pages=1, maximum=None):
    liens = []
    liens_deja_vus = set()

    for page in range(1, nombre_pages + 1):
        url_page = "https://www.lesechos.fr/recherche?search=ia&searchType=posts"
        if page > 1:
            url_page += f"&page={page}"

        driver.get(url_page)
        WebDriverWait(driver, TIMEOUT).until(
            EC.presence_of_all_elements_located((By.XPATH, "//a[.//h3]"))
        )

        for carte in driver.find_elements(By.XPATH, "//a[.//h3]"):
            url = (carte.get_attribute("href") or "").split("?")[0]
            if url and url not in liens_deja_vus:
                liens.append(url)
                liens_deja_vus.add(url)

            if maximum and len(liens) >= maximum:
                return liens

    return liens


liens_articles = recuperer_liens_articles(driver, NOMBRE_PAGES, MAX_ARTICLES)
print(f"{len(liens_articles)} article(s) collecté(s).")
liens_articles[:5]

20 article(s) collecté(s).


['https://www.lesechos.fr/idees-debats/editos-analyses/peut-on-encore-desarmer-lia-2234593',
 'https://www.lesechos.fr/idees-debats/editos-analyses/de-la-necessite-dun-choc-doffre-2234592',
 'https://www.lesechos.fr/idees-debats/sciences-prospective/mottronix-des-composants-inspires-du-cerveau-pour-une-ia-econome-en-energie-2234591',
 'https://www.lesechos.fr/partenaires/cooperation-sante/sante-les-propositions-pour-2027-se-construisent-des-maintenant-2234577',
 'https://www.lesechos.fr/finance-marches/marches-financiers/alphabet-lance-lune-des-plus-importantes-levees-de-capital-de-lhistoire-pour-financer-ses-investissements-dans-lia-2234573']

# Étape 8 - Extraire le titre et le résumé de chaque article

## Extraction sur la page de l'article

- Le titre est récupéré dans `<h1>`.
- Le petit résumé est récupéré dans `p[data-testid='post-lead']`.
- La métadonnée `meta[name='description']` sert de solution de secours, notamment pour certains articles premium.

In [8]:
def extraire_article(driver, url):
    driver.get(url)
    WebDriverWait(driver, TIMEOUT).until(
        EC.presence_of_element_located((By.TAG_NAME, "h1"))
    )

    titre = texte_premier(driver, [
        (By.TAG_NAME, "h1"),
        (By.CSS_SELECTOR, "meta[property='og:title']")
    ], valeur_par_defaut="Titre non trouvé")

    resume = texte_premier(driver, [
        (By.CSS_SELECTOR, "p[data-testid='post-lead']")
    ])

    if not resume:
        resume = attribut_premier(driver, [
            (By.CSS_SELECTOR, "meta[name='description']"),
            (By.CSS_SELECTOR, "meta[property='og:description']")
        ], "content", valeur_par_defaut="Résumé non trouvé")

    return {
        "titre": titre,
        "resume": resume,
        "url": url
    }


articles = []
for index, url in enumerate(liens_articles, start=1):
    try:
        article = extraire_article(driver, url)
        articles.append(article)
        print(f"[{index}/{len(liens_articles)}] {article['titre']}")
    except Exception as erreur:
        print(f"[{index}/{len(liens_articles)}] Article ignoré : {url} ({erreur})")

df_articles = pd.DataFrame(articles)
df_articles

[1/20] Peut-on encore désarmer l'IA ?
[2/20] De la nécessité d'un choc d'offre
[3/20] Mottronix : des composants inspirés du cerveau pour une IA économe en énergie
[4/20] Santé : les propositions pour 2027 se construisent dès maintenant
[5/20] IA : Alphabet lance l'une des plus importantes levées de capital de l'histoire
[6/20] « Nous ne pouvons pas rester les bras croisés » : l'appel du « New York Times » pour s'allier face aux géants de l'IA
[7/20] La fin d'une époque, dans les coulisses du dernier Choose France d'Emmanuel Macron
[8/20] Bull Angers au coeur d'un projet de 120 millions d'euros pour les infrastructures IA
[9/20] L'IA, miroir de nos désirs et piège à illusions
[10/20] Sans contexte ni confiance, l’IA d’entreprise atteint ses limites
[11/20] Le Paris-Saclay Cancer Cluster lance un fonds de 2,5 millions d'euros pour les start-up qui innovent contre le cancer
[12/20] La révolte gronde contre l'IA
[13/20] Stéphane Cohen veut convertir les pressings à l'IA
[14/20] Gestion d'

,titre,resume,url
0,Peut-on encore désarmer l'IA ?,L'intelligence artificielle est d'ores et déjà...,https://www.lesechos.fr/idees-debats/editos-an...
1,De la nécessité d'un choc d'offre,L'économie française ne manque pas de consomma...,https://www.lesechos.fr/idees-debats/editos-an...
2,Mottronix : des composants inspirés du cerveau...,Une start-up issue de la recherche publique dé...,https://www.lesechos.fr/idees-debats/sciences-...
3,Santé : les propositions pour 2027 se construi...,Le système de santé français est sous pression...,https://www.lesechos.fr/partenaires/cooperatio...
4,IA : Alphabet lance l'une des plus importantes...,La maison mère de Google a annoncé un plan d'é...,https://www.lesechos.fr/finance-marches/marche...
5,« Nous ne pouvons pas rester les bras croisés ...,Le patron du prestigieux journal américain a d...,https://www.lesechos.fr/tech-medias/medias/nou...
6,"La fin d'une époque, dans les coulisses du der...","A Versailles, dirigeants et ministres ont célé...",https://www.lesechos.fr/economie-france/conjon...
7,Bull Angers au coeur d'un projet de 120 millio...,Bull et Foxconn s'allient pour produire en Eur...,https://www.lesechos.fr/pme-regions/pays-de-la...
8,"L'IA, miroir de nos désirs et piège à illusions","Le coût économique, écologique et social de l'...",https://www.lesechos.fr/idees-debats/sciences-...
9,"Sans contexte ni confiance, l’IA d’entreprise ...",Après dix-huit mois d’expérimentations intensi...,https://www.lesechos.fr/partenaires/business-r...


# Étape 9 - Exporter les résultats

## Enregistrement au format CSV

Le fichier est encodé en UTF-8 avec BOM afin de conserver correctement les accents lors de son ouverture dans Excel.

In [9]:
fichier_sortie = Path("articles_les_echos_ia.csv")
df_articles.to_csv(fichier_sortie, index=False, encoding="utf-8-sig")
print("Résultats enregistrés dans :", fichier_sortie.resolve())

Résultats enregistrés dans : C:\TCHIKSON\Webscrapping\articles_les_echos_ia.csv


# Étape 10 - Fermer le navigateur


In [10]:
driver.quit()
print("Navigateur fermé.")

Navigateur fermé.
